# 🏷️ Cross-match TNS — Fink/LSST

Ce notebook explore les alertes Fink/LSST ayant un **contrepartie dans le Transient Name Server (TNS)**
via le tag `in_tns`, et les compare avec les autres tags.

Pour chaque objet `in_tns` :
- Type TNS (AT, SN Ia, SN II, SN Ib/c...)
- Nom TNS (AT20xxXXX)
- Comparaison des scores Fink entre objets confirmés et non confirmés
- Carte Mollweide colorée par type TNS
- Courbes de lumière des objets les mieux classifiés
- Tableau récapitulatif avec liens TNS + Fink portal

**API :** `https://api.lsst.fink-portal.org`  
Endpoints : `/api/v1/tags` · `/api/v1/sources` · `/api/v1/schema`

## 1. Paramètres

In [ ]:
# ─── Paramètres utilisateur ───────────────────────────────────────────────────
N_TNS        = 200        # Alertes in_tns à récupérer
N_OTHER      = 100        # Alertes par autre tag (pour comparaison)
STARTDATE    = None
STOPDATE     = None
BASE_URL     = "https://api.lsst.fink-portal.org"

# Tags de comparaison (None = tous sauf in_tns)
COMPARE_TAGS = None

# Nombre max de courbes de lumière à afficher (objets TNS avec score le plus élevé)
N_LC_DISPLAY = 5

SAVE_FIGS    = True
OUTPUT_DIR   = "tns_outputs"
# ─────────────────────────────────────────────────────────────────────────────

## 2. Imports

In [ ]:
import os
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.dates as mdates
from IPython.display import display as ipy_display, HTML
from tqdm.notebook import tqdm
from astropy.coordinates import SkyCoord
import astropy.units as u

warnings.filterwarnings('ignore')
os.makedirs(OUTPUT_DIR, exist_ok=True)

mpl.rcParams.update({
    'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.labelsize': 11, 'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.fontsize': 9, 'figure.titlesize': 13, 'figure.titleweight': 'bold',
})

FILTER_COLORS = {
    'u': '#7B2FBE', 'g': '#0077BB', 'r': '#33AA77',
    'i': '#DDAA33', 'z': '#BB5500', 'y': '#AA0000',
}
MJD_EPOCH = pd.Timestamp('1858-11-17', tz='UTC')
ZP_AB     = 2.5 * np.log10(3631e9)

def mjd_to_datetime(s):
    return MJD_EPOCH + pd.to_timedelta(pd.to_numeric(s, errors='coerce'), unit='D')

def mjd_to_datestr(m):
    try: return (MJD_EPOCH + pd.to_timedelta(float(m), unit='D')).strftime('%Y-%m-%d')
    except: return str(m)

def radec_to_mollweide(ra, dec):
    return -np.deg2rad(np.asarray(ra)-180.), np.deg2rad(np.asarray(dec))

def split_curve(x, y):
    breaks = np.where(np.abs(np.diff(x)) > np.pi*0.8)[0]+1
    return np.split(x, breaks), np.split(y, breaks)

print("Imports OK ✓")

## 3. Découverte des colonnes TNS et scores disponibles

In [ ]:
# Schéma complet
resp = requests.post(f"{BASE_URL}/api/v1/schema",
                     json={"endpoint": "/api/v1/sources"}, timeout=60)
resp.raise_for_status()
df_schema = pd.DataFrame(resp.json())

# Colonnes TNS (cross-match xm avec TNS)
tns_cols = df_schema[df_schema['name'].str.contains('tns|TNS', na=False)]['name'].tolist()
clf_cols = df_schema[df_schema['name'].str.startswith('f:clf')]['name'].tolist()
xm_cols  = df_schema[df_schema['name'].str.startswith('f:xm')]['name'].tolist()

print(f"Colonnes TNS  : {tns_cols}")
print(f"Scores clf    : {clf_cols}")
print(f"Cross-matches : {xm_cols[:8]}{'...' if len(xm_cols)>8 else ''}")

## 4. Récupération des alertes `in_tns` avec colonnes TNS et scores

In [ ]:
def fetch_tag(tag, n, startdate=None, stopdate=None):
    payload = {"tag": tag, "n": n, "output-format": "json"}
    if startdate: payload["startdate"] = startdate
    if stopdate:  payload["stopdate"]  = stopdate
    r = requests.post(f"{BASE_URL}/api/v1/tags", json=payload, timeout=60)
    r.raise_for_status()
    if not r.json(): return pd.DataFrame()
    df = pd.DataFrame(r.json())
    df.columns = [c.replace('r:','').replace('f:','') for c in df.columns]
    return df


def fetch_sources_enriched(oid_list, extra_cols):
    """Récupère sources + colonnes TNS/clf pour une liste d'objets."""
    base = "r:diaObjectId,r:diaSourceId,r:midpointMjdTai,r:band,r:psfFlux,r:psfFluxErr,r:ra,r:dec"
    cols = base + (',' + ','.join(extra_cols) if extra_cols else '')
    payload = {
        "diaObjectId"  : ",".join(str(o) for o in oid_list),
        "columns"      : cols,
        "output-format": "json",
    }
    r = requests.post(f"{BASE_URL}/api/v1/sources", json=payload, timeout=120)
    r.raise_for_status()
    if not r.json(): return pd.DataFrame()
    df = pd.DataFrame(r.json())
    df.columns = [c.replace('r:','').replace('f:','') for c in df.columns]
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='ignore')
    return df


# ── Récupération tags disponibles ─────────────────────────────────────────────
r_tags = requests.get(f"{BASE_URL}/api/v1/tags")
TAGS_DOC = r_tags.json()
ALL_TAGS = list(TAGS_DOC.keys())
OTHER_TAGS = [t for t in ALL_TAGS if t != 'in_tns'] \
             if COMPARE_TAGS is None else COMPARE_TAGS

print(f"Tags disponibles : {ALL_TAGS}")
print(f"Tags de comparaison : {OTHER_TAGS}\n")

# ── Alertes in_tns ────────────────────────────────────────────────────────────
print(f"Récupération de {N_TNS} alertes 'in_tns'...")
df_tns_tags = fetch_tag('in_tns', N_TNS, STARTDATE, STOPDATE)
id_col = next(c for c in df_tns_tags.columns if 'diaObjectId' in c)
tns_oids = df_tns_tags[id_col].astype(str).unique().tolist()
print(f"✓ {len(df_tns_tags)} alertes, {len(tns_oids)} objets uniques")

# ── Sources enrichies pour les objets TNS ─────────────────────────────────────
extra = tns_cols + clf_cols[:4]  # colonnes TNS + premiers scores clf
print(f"\nRécupération des sources enrichies...")
# Traitement par lots de 20 pour ne pas surcharger l'API
batch_size = 20
chunks = [tns_oids[i:i+batch_size] for i in range(0, len(tns_oids), batch_size)]
dfs = []
for chunk in tqdm(chunks, desc='Sources TNS', unit='batch'):
    try:
        dfs.append(fetch_sources_enriched(chunk, extra))
    except Exception as e:
        print(f"  ✗ lot {chunk[:2]}... : {e}")

df_tns = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
print(f"✓ {len(df_tns)} sources pour {df_tns['diaObjectId'].nunique()} objets TNS")

## 5. Analyse des types TNS

In [ ]:
# Colonnes TNS présentes
tns_present = [c.replace('f:','') for c in tns_cols if c.replace('f:','') in df_tns.columns]
print(f"Colonnes TNS dans les données : {tns_present}")

# Colonne de type TNS (chercher 'type' ou 'classification')
type_col = next((c for c in tns_present if 'type' in c.lower() or 'classif' in c.lower()), None)
name_col = next((c for c in tns_present if 'name' in c.lower() or 'internal' in c.lower()), None)

print(f"Colonne type TNS  : {type_col}")
print(f"Colonne nom TNS   : {name_col}")

if type_col and not df_tns[type_col].dropna().empty:
    vc = df_tns.dropna(subset=[type_col]).groupby('diaObjectId')[type_col].first().value_counts()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), layout='constrained')
    fig.suptitle("Distribution des types TNS — alertes Fink/LSST")

    # Barres horizontales
    colors = plt.cm.Set2(np.linspace(0, 1, len(vc)))
    axes[0].barh(vc.index[::-1], vc.values[::-1], color=colors[::-1],
                 edgecolor='white', linewidth=0.4)
    axes[0].set_xlabel('N objets')
    axes[0].set_title('Types TNS (par objet unique)')
    axes[0].grid(True, axis='x', alpha=0.25, linewidth=0.5)

    # Camembert
    top_n = min(8, len(vc))
    vc_top = vc.head(top_n)
    if len(vc) > top_n:
        vc_top['Autres'] = vc.iloc[top_n:].sum()
    wedges, texts, autotexts = axes[1].pie(
        vc_top.values, labels=vc_top.index, autopct='%1.1f%%',
        colors=plt.cm.Set2(np.linspace(0, 1, len(vc_top))),
        startangle=90, pctdistance=0.75)
    for t in texts: t.set_fontsize(9)
    for t in autotexts: t.set_fontsize(8)
    axes[1].set_title('Répartition (%)' )

    if SAVE_FIGS:
        fig.savefig(f"{OUTPUT_DIR}/tns_types.pdf", bbox_inches='tight', dpi=150)
        fig.savefig(f"{OUTPUT_DIR}/tns_types.png", bbox_inches='tight', dpi=150)
        print(f"✓ Sauvegardé")
    plt.show()
else:
    print("⚠️  Colonne de type TNS non disponible dans les données.")

## 6. Comparaison des scores Fink : TNS confirmés vs autres tags

On compare les distributions des scores `clf.*` entre :
- les objets **`in_tns`** (avec contrepartie TNS confirmée)
- les objets des **autres tags** (candidats non encore confirmés)

In [ ]:
clf_present = [c.replace('f:','') for c in clf_cols if c.replace('f:','') in df_tns.columns]

# Récupérer les autres tags pour comparaison
other_dfs = {}
for tag in tqdm(OTHER_TAGS, desc='Tags comparaison', unit='tag'):
    try:
        df_tag = fetch_tag(tag, N_OTHER, STARTDATE, STOPDATE)
        if df_tag.empty: continue
        tag_id_col = next((c for c in df_tag.columns if 'diaObjectId' in c), None)
        if tag_id_col is None: continue
        tag_oids = df_tag[tag_id_col].astype(str).unique().tolist()
        chunks = [tag_oids[i:i+20] for i in range(0, min(len(tag_oids), 60), 20)]
        parts = []
        for chunk in chunks:
            try: parts.append(fetch_sources_enriched(chunk, clf_cols[:4]))
            except: pass
        if parts:
            other_dfs[tag] = pd.concat(parts, ignore_index=True)
    except Exception as e:
        print(f"  ✗ {tag} : {e}")

if not clf_present:
    print("⚠️  Aucun score clf disponible.")
else:
    n_clf = len(clf_present)
    n_groups = 1 + len(other_dfs)   # TNS + autres tags

    for clf_col in clf_present:
        fig, ax = plt.subplots(figsize=(10, 4), layout='constrained')
        short = clf_col.replace('clf.','')
        fig.suptitle(f"Distribution score '{short}' : TNS vs autres tags")

        # in_tns
        vals_tns = pd.to_numeric(df_tns[clf_col], errors='coerce').dropna()
        if not vals_tns.empty:
            ax.hist(vals_tns, bins=25, range=(0,1), alpha=0.6,
                    color='#EE6677', label=f"in_tns (n={len(vals_tns)})",
                    density=True, edgecolor='white', linewidth=0.3)

        # Autres tags
        tag_palette = ['#4477AA','#228833','#CCBB44','#66CCEE','#AA3377']
        for idx_t, (tag, df_ot) in enumerate(other_dfs.items()):
            if clf_col not in df_ot.columns: continue
            vals_ot = pd.to_numeric(df_ot[clf_col], errors='coerce').dropna()
            if vals_ot.empty: continue
            short_tag = tag.replace('_candidate','').replace('extragalactic_','ext_')
            ax.hist(vals_ot, bins=25, range=(0,1), alpha=0.45,
                    color=tag_palette[idx_t % len(tag_palette)],
                    label=f"{short_tag} (n={len(vals_ot)})",
                    density=True, histtype='step', linewidth=1.5)

        ax.axvline(0.5, color='black', lw=1.0, ls='--', alpha=0.6)
        ax.set_xlabel('Score')
        ax.set_ylabel('Densité')
        ax.set_xlim(0, 1)
        ax.legend(fontsize=8, ncol=2)
        ax.grid(True, axis='y', alpha=0.2, linewidth=0.5)

        if SAVE_FIGS:
            fig.savefig(f"{OUTPUT_DIR}/tns_score_comp_{short}.pdf", bbox_inches='tight', dpi=150)
            fig.savefig(f"{OUTPUT_DIR}/tns_score_comp_{short}.png", bbox_inches='tight', dpi=150)
        plt.show()

## 7. Carte Mollweide — colorée par type TNS

In [ ]:
# Position médiane par objet
df_pos = df_tns.groupby('diaObjectId').agg(
    ra=('ra', 'median'), dec=('dec', 'median')
).reset_index()

# Joindre le type TNS si disponible
if type_col and type_col in df_tns.columns:
    df_type = df_tns.dropna(subset=[type_col]).groupby('diaObjectId')[type_col].first().reset_index()
    df_pos  = df_pos.merge(df_type, on='diaObjectId', how='left')
    df_pos[type_col] = df_pos[type_col].fillna('Unknown')
    type_values = df_pos[type_col].unique()
    type_cmap   = dict(zip(sorted(type_values),
                           plt.cm.Set1(np.linspace(0, 1, len(type_values)))))
else:
    df_pos['_type'] = 'in_tns'
    type_col = '_type'
    type_cmap = {'in_tns': '#EE6677'}

# Plan galactique
l = np.linspace(0, 360, 1000)
gc = SkyCoord(l=l*u.deg, b=np.zeros(1000)*u.deg, frame='galactic').icrs
idx = np.argsort(gc.ra.deg)
gal_x, gal_y = radec_to_mollweide(gc.ra.deg[idx], gc.dec.deg[idx])

fig = plt.figure(figsize=(16, 8), layout='constrained')
ax  = fig.add_subplot(111, projection='mollweide')
fig.suptitle(f"Carte céleste — alertes in_tns ({len(df_pos)} objets)")
ax.grid(True, color='gray', alpha=0.3, linewidth=0.5, linestyle='--')

for sx, sy in zip(*split_curve(gal_x, gal_y)):
    ax.plot(sx, sy, color='goldenrod', lw=1.2)

handles = [plt.Line2D([0],[0], color='goldenrod', lw=1.2, label='Plan galactique')]

for ttype, color in type_cmap.items():
    sub = df_pos[df_pos[type_col] == ttype].dropna(subset=['ra','dec'])
    if sub.empty: continue
    px, py = radec_to_mollweide(sub['ra'].values, sub['dec'].values)
    ax.scatter(px, py, s=20, color=color, alpha=0.8, zorder=5,
               edgecolors='white', linewidths=0.3)
    handles.append(plt.Line2D([0],[0], marker='o', color='w',
                               markerfacecolor=color, markersize=7,
                               label=f"{ttype} ({len(sub)})"))

ax.legend(handles=handles, loc='lower left', fontsize=8,
          framealpha=0.75, ncol=2)

if SAVE_FIGS:
    fig.savefig(f"{OUTPUT_DIR}/tns_skymap.pdf", bbox_inches='tight', dpi=150)
    fig.savefig(f"{OUTPUT_DIR}/tns_skymap.png", bbox_inches='tight', dpi=150)
    print(f"✓ Sauvegardé")
plt.show()

## 8. Courbes de lumière des meilleurs candidats TNS

Les `N_LC_DISPLAY` objets `in_tns` avec le score `clf` le plus élevé.

In [ ]:
def flux_to_mag(flux, flux_e):
    flux, flux_e = np.asarray(flux, float), np.asarray(flux_e, float)
    v = flux > 0
    mag, me = np.full(flux.shape, np.nan), np.full(flux.shape, np.nan)
    mag[v] = -2.5*np.log10(flux[v]) + ZP_AB
    me[v]  = (2.5/np.log(10)) * np.abs(flux_e[v]/flux[v])
    return mag, me

def _date_axis(ax):
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
    ax.grid(True, alpha=0.2, linewidth=0.5)


# Sélection des meilleurs objets par score médian
if clf_present:
    df_med = df_tns.groupby('diaObjectId')[clf_present[0]].median().sort_values(ascending=False)
    top_oids = df_med.head(N_LC_DISPLAY).index.tolist()
else:
    top_oids = df_tns['diaObjectId'].unique()[:N_LC_DISPLAY].tolist()

for oid in top_oids:
    lc = df_tns[df_tns['diaObjectId'].astype(str) == str(oid)].copy()
    for col in ['midpointMjdTai','psfFlux','psfFluxErr']:
        lc[col] = pd.to_numeric(lc[col], errors='coerce')
    lc = lc.dropna(subset=['midpointMjdTai','psfFlux','psfFluxErr'])
    lc['date'] = mjd_to_datetime(lc['midpointMjdTai'])

    # Infos TNS
    tns_name = lc[name_col].dropna().iloc[0] if (name_col and name_col in lc.columns
                and not lc[name_col].dropna().empty) else '—'
    tns_type = lc[type_col].dropna().iloc[0] if (type_col and type_col in lc.columns
                and not lc[type_col].dropna().empty) else '—'
    score_str = ''
    if clf_present:
        s = pd.to_numeric(lc[clf_present[0]], errors='coerce').median()
        score_str = f"   score {clf_present[0].replace('clf.','')} : {s:.3f}"

    fig, (ax_f, ax_m) = plt.subplots(1, 2, figsize=(14, 4), layout='constrained')
    fig.suptitle(f"diaObjectId : {oid}   |   TNS : {tns_name}   type : {tns_type}{score_str}")

    for band in sorted(lc['band'].dropna().unique()):
        sub   = lc[lc['band']==band].sort_values('date')
        color = FILTER_COLORS.get(band,'#888')
        ax_f.errorbar(sub['date'], sub['psfFlux'], yerr=sub['psfFluxErr'],
                      fmt='o', color=color, label=f"${band}$",
                      ms=4, capsize=3, capthick=0.8, elinewidth=0.8, alpha=0.85)
        mag, me = flux_to_mag(sub['psfFlux'].values, sub['psfFluxErr'].values)
        det = ~np.isnan(mag)
        if det.any():
            ax_m.errorbar(sub['date'].values[det], mag[det], yerr=me[det],
                          fmt='o', color=color, label=f"${band}$",
                          ms=4, capsize=3, capthick=0.8, elinewidth=0.8, alpha=0.85)

    ax_f.axhline(0, color='gray', lw=0.6, ls='--', alpha=0.5)
    ax_f.set_ylabel('Flux PSF (nJy)')
    ax_f.legend(ncol=len(lc['band'].dropna().unique()), loc='upper left', handlelength=1)
    _date_axis(ax_f)
    ax_m.invert_yaxis()
    ax_m.set_ylabel('Magnitude AB')
    ax_m.legend(ncol=len(lc['band'].dropna().unique()), loc='upper left', handlelength=1)
    _date_axis(ax_m)

    if SAVE_FIGS:
        fig.savefig(f"{OUTPUT_DIR}/tns_lc_{oid}.pdf", bbox_inches='tight', dpi=150)
        fig.savefig(f"{OUTPUT_DIR}/tns_lc_{oid}.png", bbox_inches='tight', dpi=150)
    plt.show()

    fink_url = f"https://lsst.fink-portal.org/{oid}"
    ipy_display(HTML(
        f'<div style="font-size:12px;margin:2px 0 14px 4px;">'
        f'🔗 <b>Fink</b> : <a href="{fink_url}" target="_blank">{fink_url}</a>'
        + (f'&nbsp;&nbsp;|&nbsp;&nbsp;🏷️ <b>TNS</b> : '
           f'<a href="https://www.wis-tns.org/search?name={tns_name}" target="_blank">'
           f'{tns_name}</a>' if tns_name != '—' else '') +
        f'</div>'
    ))
    print()

## 9. Tableau récapitulatif — objets TNS

In [ ]:
agg = {'ra': 'median', 'dec': 'median', 'psfFlux': 'max',
       'midpointMjdTai': 'max', 'band': lambda x: ', '.join(sorted(x.dropna().unique()))}
if type_col and type_col in df_tns.columns:
    agg[type_col] = 'first'
if name_col and name_col in df_tns.columns:
    agg[name_col] = 'first'
for c in clf_present:
    agg[c] = 'median'

df_summary = df_tns.groupby('diaObjectId').agg(agg).reset_index()
df_summary['last_date'] = df_summary['midpointMjdTai'].apply(mjd_to_datestr)
df_summary['ra']  = df_summary['ra'].round(5)
df_summary['dec'] = df_summary['dec'].round(5)
df_summary['psfFlux'] = df_summary['psfFlux'].round(1)
for c in clf_present:
    df_summary[c] = df_summary[c].round(3)

if clf_present:
    df_summary = df_summary.sort_values(clf_present[0], ascending=False)

# Liens
df_summary['Fink'] = df_summary['diaObjectId'].apply(
    lambda o: f'<a href="https://lsst.fink-portal.org/{o}" target="_blank">🔗 {o}</a>')
if name_col and name_col in df_summary.columns:
    df_summary['TNS'] = df_summary[name_col].apply(
        lambda n: f'<a href="https://www.wis-tns.org/search?name={n}" target="_blank">🏷️ {n}</a>'
        if pd.notna(n) and n != '' else '—')

html = df_summary.drop(columns=['midpointMjdTai'], errors='ignore')\
                 .to_html(index=False, escape=False, border=0, classes='fink-table')
style = """
<style>
  .fink-table { border-collapse:collapse; font-size:12px; width:100%; }
  .fink-table th { background:#f0f0f0; padding:6px 10px;
                   border-bottom:2px solid #ccc; text-align:left; }
  .fink-table td { padding:4px 10px; border-bottom:1px solid #eee;
                   text-align:left; white-space:nowrap; }
  .fink-table tr:hover td { background:#fafafa; }
</style>
"""
ipy_display(HTML(style + html))